# Task 1

In [1]:
import pandas as pd
import sqlite3
conn = sqlite3.connect("level_3_final_project_library.db")

In [2]:
pd.read_sql_query("SELECT * FROM members",conn)

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05
...,...,...,...,...,...,...,...
75,1076,Dina,Wahba,7.0,Shubra,Active,2025-12-13
76,1077,Lina,Rashad,6.0,Shubra,Active,2023-07-08
77,1078,Habiba,Osman,7.0,Shubra,INACTIVE,2023-08-05
78,1079,Rana,Osman,8.0,Shubra,Active,2024-10-27


In [3]:
pd.read_sql_query("SELECT * FROM books",conn)

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez
5,506,The Paper Boat Club,Aya Hafez
6,507,Fossils and Fireflies,Dalia Serry
7,508,The Quiet Observatory,Dalia Serry
8,509,Marbles and Mirrors,Diaa Sultan
9,510,The Missing Metronome,Diaa Sultan


In [4]:
pd.read_sql_query("SELECT * FROM checkouts",conn)

,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03
...,...,...,...,...,...
386,9232,1044,513,2025-05-26,2025-06-11
387,9084,1008,511,2024-06-27,2024-07-22
388,9116,1024,519,2025-01-10,2025-02-09
389,9352,1076,501,2025-02-02,None


In [5]:
members = pd.read_sql_query('SELECT * FROM members', conn)
checkouts = pd.read_sql_query('SELECT * FROM checkouts', conn)
conn.close()

In [6]:
checkout_counts = checkouts.groupby('member_id').size().reset_index(name='checkout_counts')
merged = pd.merge(members, checkouts, on='member_id', how='outer')
merged = pd.merge(merged, checkout_counts, on='member_id', how='left')
merged['checkout_counts'] = merged['checkout_counts'].fillna(0).astype(int)

In [ ]:
books = pd.read_json('level_3_final_project_book_catalog.json')

In [ ]:
full_merge = pd.merge(merged, books, on='book_id', how='left')

In [ ]:
full_merge['source'] = full_merge['checkout_id'].notna().map({True: 'Database', False: pd.NA})

In [ ]:
kickoff_raw = pd.read_html('level_3_final_project_event_signup.html')[0]
kickoff_raw.columns = ['member_id', 'book_id', 'checkout_date']
kickoff_raw['checkout_id'] = pd.NA
kickoff_raw['return_date'] = pd.NA
kickoff_raw['source'] = 'Kickoff'

In [ ]:
kickoff_with_members = pd.merge(kickoff_raw, members, on='member_id', how='left')
kickoff_with_members = pd.merge(kickoff_with_members, checkout_counts, on='member_id', how='left')
kickoff_with_members['checkout_counts'] = kickoff_with_members['checkout_counts'].fillna(0).astype(int)
kickoff_with_members = pd.merge(kickoff_with_members, books, on='book_id', how='left')
kickoff_with_members = kickoff_with_members[full_merge.columns]

In [ ]:
final_merge = pd.concat([full_merge, kickoff_with_members], ignore_index=True)
final_merge.to_csv('task1_combined_data.csv', index=False)

# Task 2

In [ ]:
df = pd.read_csv('task1_combined_data.csv')

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df = df.drop_duplicates(keep='first')

In [ ]:
def normalize_text(series):
    s = series.astype('string')
    s = s.str.strip()
    s = s.str.replace(r'\s+', ' ', regex=True)
    s = s.str.title()
    return s

df['neighborhood'] = normalize_text(df['neighborhood'])
df['membership_status'] = normalize_text(df['membership_status'])

In [ ]:
conn = sqlite3.connect('level_3_final_project_library.db')
members_check = pd.read_sql_query('SELECT member_id FROM members', conn)
conn.close()
registered_ids = set(members_check['member_id'])

orphan_mask = ~df['member_id'].isin(registered_ids)
df = df[~orphan_mask].copy()

In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col, stat in [('grade', 'median'), ('pages', 'mean'), ('publication_year', 'median')]:
    s = df[col].dropna()
    lo, hi = iqr_bounds(s)
    s_no_outliers = s[(s >= lo) & (s <= hi)]
    fill_value = s_no_outliers.median() if stat == 'median' else s_no_outliers.mean()
    df[col] = df[col].fillna(round(fill_value, 1) if stat == 'mean' else fill_value)

In [ ]:
for col in ['genre', 'publisher']:
    df[col] = df[col].fillna(df[col].mode().iloc[0])

In [ ]:
df = df.sort_values('member_id')
df['join_date'] = df['join_date'].ffill()
df = df.sort_index()

In [ ]:
df.to_csv('task2_cleaned_data.csv', index=False)

In [ ]:
df = pd.read_csv('task2_cleaned_data.csv')

In [ ]:
members_df = df.drop_duplicates(subset='member_id')[['member_id', 'neighborhood']]
members_per_neigh = members_df.groupby('neighborhood')['member_id'].nunique()

In [ ]:
checkouts_only = df[df['source'].notna()]
checkouts_per_neigh = checkouts_only.groupby('neighborhood').size()

In [ ]:
summary = pd.DataFrame({
    'members': members_per_neigh,
    'checkouts': checkouts_per_neigh,
}).fillna(0)
summary['pct_of_members'] = (summary['members'] / summary['members'].sum() * 100).round(1)
summary['pct_of_checkouts'] = (summary['checkouts'] / summary['checkouts'].sum() * 100).round(1)
summary['checkouts_per_member'] = (summary['checkouts'] / summary['members']).round(2)
summary.sort_values('checkouts_per_member', ascending=False)